# Lily 1.5B GRPO Model Inference — Google Colab
**Interactive Reasoning & Inference for `abhinav0231/Lily-1.5B`**

This notebook runs in Google Colab. It loads the trained GRPO model `abhinav0231/Lily-1.5B` using Unsloth's 2x fast inference engine (`FastLanguageModel.for_inference`), parses step-by-step thinking `<think>...</think>` tags and final `<answer>...</answer>` tags, and includes an interactive query loop for testing.

## Cell 1 — Install Unsloth & Dependencies

In [1]:
# ==============================================================================
# Cell 1 — Install Unsloth & Hugging Face Libraries
# ==============================================================================
# Install Unsloth fast inference engine and huggingface_hub for model downloading
!pip install unsloth huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

## Cell 2 — Load Trained GRPO Model with Unsloth

In [3]:
# ==============================================================================
# Cell 2 — Authentication & Load Trained GRPO Model with Unsloth
# ==============================================================================
import os
import torch
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# ── Load Model with Unsloth FastLanguageModel (2x Faster Inference) ───────────
from unsloth import FastLanguageModel

MODEL_REPO  = "abhinav0231/Lily-1.5B-v0.1"
MAX_SEQ_LEN = 3072

print(f"Loading {MODEL_REPO} (max_seq_len={MAX_SEQ_LEN}, 4-bit quantized)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_REPO,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE" else None,
)

# Enable native Unsloth fast inference mode
FastLanguageModel.for_inference(model)
print("✅ Lily-1.5B model and tokenizer loaded successfully for fast inference")


✅ Authenticated with Hugging Face
Loading abhinav0231/Lily-1.5B-v0.1 (max_seq_len=3072, 4-bit quantized)...
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Lily-1.5B model and tokenizer loaded successfully for fast inference


## Cell 3 — Inference Function with Reasoning CoT Parser

In [4]:
# ==============================================================================
# Cell 3 — Prompt Formatting, Generation Loop & Regular Expression CoT Parser
# ==============================================================================
import re

# Standardized System Prompt to trigger structural step-by-step reasoning
SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

def ask(question, max_new_tokens=1024, temperature=0.7):
    """
    Formats user prompt into ChatML structure, tokenizes, generates response via PyTorch,
    and returns newly generated completion string.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    # Format and tokenize input query with generation prompt prompt delimiter
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    # Generate model response using sampling parameters
    output_ids = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = 0.95,
        do_sample      = True if temperature > 0 else False,
        pad_token_id   = tokenizer.eos_token_id,
    )

    # Decode only the newly generated tokens (excluding prompt tokens)
    response = tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:],
        skip_special_tokens = True
    )
    return response

def parse_and_print(response):
    """
    Extracts and prints 'think_body' (<think>...</think>) and 'answer_body' (<answer>...</answer>)
    using regular expressions.
    """
    think_m  = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)

    print("REASONING (<think>):")
    if think_m:
        print(think_m.group(1).strip())
    else:
        print("No <think> tag found. Full output:")
        print(response)

    print("\nFINAL ANSWER (<answer>):")
    if answer_m:
        print(answer_m.group(1).strip())
    else:
        print(response.split("</think>")[-1].strip())

print("✅ Inference & Parsing functions ready")

✅ Inference & Parsing functions ready


## Cell 4 — Run Sample Test Queries (Math, Logic, Coding)

In [5]:
# ==============================================================================
# Cell 4 — Test Execution Across Mathematics, Physics & Coding Tasks
# ==============================================================================
test_questions = [
    "What is 15% of 840?",
    "If a train travels 120 km in 1.5 hours, what is its speed in m/s?",
    "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much does the ball cost?",
    "Write a Python function to check if a string is a palindrome."
]

# Execute generation and parse output tags for each test query
for q in test_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {q}")
    print(f"{'='*70}")
    raw_out = ask(q, temperature=0.7)
    parse_and_print(raw_out)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



❓ QUESTION: What is 15% of 840?


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 REASONING (<think>):
To find 15% of 840, we can follow these steps:

1. **Convert the percentage to a decimal**:
   - 15% is equivalent to \(\frac{15}{100}\), which simplifies to 0.15.

2. **Multiply the decimal by the number**:
   - Multiply 0.15 by 840.

\[
0.15 \times 840
\]

3. **Perform the multiplication**:
   - \(0.15 \times 840 = 126\)

Therefore, 15% of 840 is \(\boxed{126}\).

🎯 FINAL ANSWER (<answer>):
126

❓ QUESTION: If a train travels 120 km in 1.5 hours, what is its speed in m/s?


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 REASONING (<think>):
To find the speed of the train in meters per second (m/s), we need to follow these steps:

1. **Calculate the speed in kilometers per hour (km/h):**
   The train travels 120 kilometers in 1.5 hours. So, we can calculate the speed as follows:
   \[
   \text{Speed in km/h} = \frac{120 \text{ km}}{1.5 \text{ hours}}
   \]
   Simplifying this:
   \[
   \text{Speed in km/h} = \frac{120}{1.5} = 80 \text{ km/h}
   \]

2. **Convert the speed from kilometers per hour to meters per second:**
   We know that 1 kilometer is equal to 1000 meters and 1 hour is equal to 3600 seconds. Therefore, to convert km/h to m/s, we use the conversion factor:
   \[
   \text{Speed in m/s} = \text{Speed in km/h} \times \frac{1000 \text{ m}}{1 \text{ km}} \times \frac{1 \text{ hour}}{3600 \text{ s}}
   \]
   Simplifying this:
   \[
   \text{Speed in m/s} = 80 \text{ km/h} \times \frac{1000}{3600} \text{ m/s}
   \]
   Simplify the fraction:
   \[
   \frac{1000}{3600} = \frac{10}{36} = \frac{5}

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 REASONING (<think>):
To solve this problem, we can set up a system of equations based on the information given.

Let's define:
- \( B \) as the cost of the ball.
- \( W \) as the cost of the bat.

From the problem, we have two pieces of information:
1. The total cost of the bat and the ball is $1.10.
2. The bat costs $1.00 more than the ball.

We can translate these statements into equations:
1. \( B + W = 1.10 \)
2. \( W = B + 1.00 \)

We can substitute the second equation into the first equation to solve for \( B \):
\[ B + (B + 1.00) = 1.10 \]

Simplify the equation:
\[ 2B + 1.00 = 1.10 \]

Subtract 1.00 from both sides:
\[ 2B = 0.10 \]

Divide both sides by 2:
\[ B = 0.05 \]

So, the ball costs $0.05.

To verify:
- The bat costs $0.05 + $1.00 = $1.05.
- The total cost is $1.05 + $0.05 = $1.10, which matches the problem statement.

Therefore, the cost of the ball is \(\boxed{0.05}\).

🎯 FINAL ANSWER (<answer>):
0.05

❓ QUESTION: Write a Python function to check if a string is a pa

## Cell 5 — Interactive Query Testing Loop

In [ ]:
# ==============================================================================
# Cell 5 — Interactive User Query Loop
# ==============================================================================
print("Type your question below (or type 'exit' to stop):\n")
while True:
    user_query = input("Enter Query: ")
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("Exiting interactive loop.")
        break
    if not user_query.strip():
        continue

    print(f"\n{'='*70}")
    print(f"QUESTION: {user_query}")
    print(f"{'='*70}")
    raw_out = ask(user_query, temperature=0.7)
    parse_and_print(raw_out)
    print("\n")

Type your question below (or type 'exit' to stop):

Enter Query: i want to get my car washed at the car washer place which is just 50 meters away. should i walk or drive to that place?


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ QUESTION: i want to get my car washed at the car washer place which is just 50 meters away. should i walk or drive to that place?
🧠 REASONING (<think>):
To determine whether it's better to walk or drive to the car wash place, we need to consider the distance and the context of the task. Here's a step-by-step analysis:

1. **Distance**: The car wash is 50 meters away. This is a relatively short distance, and walking is generally quicker than driving.

2. **Context of the Task**: You want to get your car washed at the car wash place. The choice between walking and driving depends on the time it takes to walk and the ease of parking.

3. **Walking**:
   - **Pros**:
     - Faster: Walking is quicker than driving.
     - Less stressful: Walking is a more active and less stressful form of transportation.
     - Environmental benefits: Walking is better for the environment compared to driving.

4. **Driving**:
   - **Pros**:
     - Potentially more efficient: Driving can be faster, especia